In [1]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from anngeno import AnnGeno
import multiprocessing

from plotnine import *
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

num_cores = multiprocessing.cpu_count()
print(num_cores)

32


In [2]:
# Configuration and paths
corr_method = 'spearman'
trait_type = 'quantitative'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'
mac = 100

maf = 1e-3
n = !wc -l $eur_samples_path
n_eur = int(n[0].split(' ')[0])
mac = maf * (2 * n_eur)
mac

config_path = f'/home/dnanexus/ukbgym/config_wgs.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

# Collect annotation categories
all_annotation_list = []
rare_variant_annotations_dict = config.get("rare_variant_annotations")
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

In [3]:
# Strategy 1: Filter first, then collect smaller chunks
print("Loading and filtering appv...")
appv = pl.scan_parquet("/home/dnanexus/data_dir/avg_pheno_per_var_quantitative_EUR_genebass1e6.parquet")
appv = appv.filter(pl.col('n_individuals') <= mac)

# Get valid IDs and phenotypes with streaming
valid_ids = appv.select(pl.col('id').unique()).collect(engine='streaming').to_series().to_list()
valid_phenos = appv.select(pl.col('phenotype').unique()).collect(engine='streaming').to_series().to_list()

print(f"Valid IDs: {len(valid_ids)}, Valid phenotypes: {len(valid_phenos)}")

# Load annotations with heavy filtering first
print("Loading and filtering annotations...")
anno = pl.scan_parquet("/home/dnanexus/data_dir/cadd_vep_annotations_processed_final_selected.parquet")
existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]

# Filter annotations by valid IDs first (this should reduce size significantly)
anno_filtered = anno.filter(
    pl.col('id').is_in(valid_ids)
).select(
    ['id', 'region'] + existing_annos
)

Loading and filtering appv...
Valid IDs: 54815851, Valid phenotypes: 127
Loading and filtering annotations...


In [4]:
# Strategy 2: Process annotations in chunks to avoid memory issues
def process_anno_chunk(chunk_df, existing_annos):
    """Process a chunk of annotations with unpivot"""
    return chunk_df.unpivot(
        index=["id", "region"],
        on=existing_annos,
        variable_name="annotation",
        value_name="annotation_score"
    ).with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )

# Strategy 3: Process in chunks without counting first
print("Processing annotations in chunks...")

chunk_size = 200_000  # Start smaller to be safe
melted_chunks = []
offset = 0

print("Processing chunks until exhausted...")
while True:
    try:
        # Try to get a chunk
        chunk = (
            anno_filtered
                .slice(offset, chunk_size)
                .collect(engine='streaming')
            )
        
        # If chunk is empty, we've reached the end
        if len(chunk) == 0:
            print(f"Finished processing at offset {offset}")
            break
            
        print(f"Processing chunk at offset {offset}, size: {len(chunk)}")
        
        # Process the chunk
        melted_chunk = process_anno_chunk(chunk, existing_annos)
        melted_chunks.append(melted_chunk)
        
        # Clean up
        del chunk
        gc.collect()
        
        # Move to next chunk
        offset += chunk_size
        
    except Exception as e:
        print(f"Error processing chunk at offset {offset}: {e}")
        # Try with smaller chunks if we hit memory issues
        if chunk_size > 100_000:
            chunk_size = chunk_size // 2
            print(f"Reducing chunk size to {chunk_size}")
            continue
        else:
            raise e

# Concatenate all chunks
if melted_chunks:
    print(f"Concatenating {len(melted_chunks)} annotation chunks...")
    melted_anno = pl.concat(melted_chunks, how="vertical")
    print(f"Final melted annotations shape: {melted_anno.shape}")
    
    # Clean up
    del melted_chunks
    gc.collect()
else:
    print("No chunks were processed!")
    raise ValueError("Failed to process any annotation chunks")

print(f"Melted annotations shape: {melted_anno.shape}")

Processing annotations in chunks...
Processing chunks until exhausted...


: 

In [ ]:
# Rest of your code continues...
a = pl.read_parquet('/home/dnanexus/data_dir/genebass_continuous_associations_ukbbgym.pq').with_columns(
    (pl.col('description').str.replace_all(r"\(", "").str.replace_all(r"\)", "").str.replace_all(' ', '_').str.to_lowercase() + '_int').alias('phenotype'),
    pl.col('gene_id').alias('region'),
    pl.col('gene_symbol').alias('gene_name')
).filter(
    (pl.col('annotation').str.contains('pLoF')) &
    (pl.col('phenotype').is_in(valid_phenos)) &
    (pl.col('region').is_in(melted_anno['region'].unique()))
)

gene_trait_df = a[['region', 'gene_id', 'gene_name', 'phenotype', 'trait_type']].unique()

# Get beta directions
plof = pl.read_parquet('/home/dnanexus/data_dir/rvat_EUR_500k_regenie.parquet').with_columns(
    (pl.col('trait') + '_int').alias('phenotype'),
    pl.col('gene_id').alias('region')
)
gene_trait_df = gene_trait_df.join(plof[['region', 'phenotype', 'beta']], on=['region', 'phenotype'], how='inner')

# Filter annotations and appv to those in gene_trait_df
anno_flt = melted_anno.filter(
    pl.col('region').is_in(gene_trait_df['region'].unique())
)

appv_flt = appv.filter(
    pl.col('id').is_in(anno_flt['id'].unique())
).drop('std_pheno_value').drop_nulls().collect(engine='streaming')

print(f"Filtered shapes - anno: {anno_flt.shape}, appv: {appv_flt.shape}")

# Join all dataframes
pheno_anno = (
    appv_flt.lazy()
    .join(anno_flt.lazy(), on="id", how="inner")
    .join(gene_trait_df.lazy(), on=["region", "phenotype"], how="inner")
    .collect(engine='streaming')
)

print(f"Final joined shape: {pheno_anno.shape}")

# Rank the mean phenotype values and annotation scores
rank_expressions = [
    pl.col(c)
    .rank("ordinal")
    .over(["region", "phenotype", "annotation"])
    .alias(f"{c}_rank")
    for c in ['mean_pheno_value', 'annotation_score']
]

# Apply the window expressions
pheno_anno = pheno_anno.with_columns(rank_expressions)

correlation_df = (
    pheno_anno
    .group_by(["region", "gene_name", "phenotype", "annotation"])
    .agg(
        pl.col("id").count().alias("n_variants"),
        pl.corr(pl.col("mean_pheno_value_rank"), pl.col("annotation_score_rank")).alias("correlation")
    ).with_columns(
        pl.col("correlation").abs().alias("abs_corr"),
    )
)

# Check if the variant counts are consistent within each (region, phenotype) pair
inconsistent = (
    correlation_df
    .group_by(["region", "phenotype"])
    .agg(pl.col("n_variants").n_unique().alias("n_variants_unique"))
    .filter(pl.col("n_variants_unique") > 1)
)
print(f"Inconsistent (region, phenotype) pairs: {inconsistent.shape[0]}")